In [1]:
import psycopg2
import re
import os
from dotenv import load_dotenv

load_dotenv()

def get_conn():
    return psycopg2.connect(
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        host=os.getenv("DB_HOST")
    )

In [6]:
INDEX_AUTHORS = "CREATE INDEX on books (authors, average_rating)"
INDEX_NUM_PAGES = "CREATE INDEX on books (num_pages)"
INDEX_RATING = "CREATE INDEX on books (average_rating)"

In [5]:
def explain_query(sql, params=None):
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("EXPLAIN ANALYZE " + sql, params)
    rows = cur.fetchall()
    cur.close()
    conn.close()
    return "\n".join(row[0] for row in rows)

def parse_explain(plain_text):
    results = {}

    # Look for the scan type in the Explain
    if "Index Scan" in plain_text:
        results["scan_type"] = "Index Scan"
    elif "Bitmap Heap Scan" in plain_text:
        results["scan_type"] = "Bitmap Heap Scan"
    else:
        results["scan_type"] = "Sequential Scan"

    # Look for the Execution time in the Explain
    time_match = re.search(r"Execution Time: ([\d.]+) ms", plain_text)
    if time_match:
        results["execution_time"] = float(time_match.group(1))

    return results

def compare_indexes(sql, params, index_sql):
    conn = get_conn()
    cur = conn.cursor()

    # Before indexing
    before_plan = explain_query(sql, params)
    before = parse_explain(before_plan)

    # Create Index
    cur.execute(index_sql)
    conn.commit()

    # After Indexing
    after_plan = explain_query(sql, params)
    after = parse_explain(after_plan)

    cur.close()
    conn.close()

    return {
        "before": before,
        "after": after,
        "before_plan": before_plan,
        "after_plan": after_plan
    }

In [7]:
if __name__ == "__main__":
    conn = get_conn()
    print("Connected")
    conn.close()

Connected
